# Vision Pipeline Run

Run the vision training pipeline (deepfake, brand, temporal).

Steps:
- Run the vision training wrapper.
- Review vision metrics outputs.
- Verify saved model artifacts.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'metrics': {},
    'artifacts': {},
    'plots': [],
}

run([PY, 'scripts/train_all_vision.py'])


In [ ]:
# Load vision metrics.
import pandas as pd

metrics_paths = [
    REPO_ROOT / 'experiments' / 'vision' / 'metrics.csv',
    REPO_ROOT / 'experiments' / 'vision' / 'video_temporal' / 'metrics.json',
    REPO_ROOT / 'experiments' / 'vision' / 'temporal_lstm' / 'metrics.json',
]
for path in metrics_paths:
    if not path.exists():
        continue
    if path.suffix == '.json':
        data = json.loads(path.read_text(encoding='utf-8'))
        summary['metrics'][str(path.relative_to(REPO_ROOT))] = data
        print(path.name, data)
    else:
        df = pd.read_csv(path)
        summary['metrics'][str(path.relative_to(REPO_ROOT))] = {
            'rows': int(df.shape[0]),
            'cols': int(df.shape[1]),
        }
        print(path.name, df.head(5))


In [ ]:
# Verify model artifacts.
artifact_paths = [
    REPO_ROOT / 'models' / 'vision',
    REPO_ROOT / 'models' / 'vision' / 'video_temporal_model.pkl',
    REPO_ROOT / 'artifacts' / 'brand' / 'yolo_logo_det.pt',
]
for path in artifact_paths:
    if path.is_dir():
        items = [p.name for p in path.iterdir()]
        summary['artifacts'][str(path.relative_to(REPO_ROOT))] = items
        print(path.relative_to(REPO_ROOT), 'items:', len(items))
    elif path.exists():
        summary['artifacts'][str(path.relative_to(REPO_ROOT))] = {
            'size_mb': round(path.stat().st_size / 1024**2, 2)
        }
        print(path.relative_to(REPO_ROOT), summary['artifacts'][str(path.relative_to(REPO_ROOT))])
    else:
        print('Missing:', path)

plots_dir = REPO_ROOT / 'experiments' / 'vision' / 'plots'
if plots_dir.exists():
    summary['plots'] = [str(p.relative_to(REPO_ROOT)) for p in plots_dir.iterdir() if p.is_file()]
    for item in summary['plots']:
        print(' -', item)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'execution_vision_pipeline_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
